# Sistema de detecção de intrusão usando uma rede multilayer Perceptron

- **A1:** [João Pedro Fernandes de Aquino](https://github.com/Joaof14)
- **A2:** [Anderson Carlos da Silva Morais](https://github.com/AndersonCSM)








## Contextualização do Projeto 

A segurança de redes de computadores é um dos desafios centrais da infraestrutura digital moderna. Ataques como negação de serviço (DoS), varredura de portas (Probe) e acesso remoto não autorizado (R2L) causam prejuízos bilionários anualmente e comprometem desde sistemas corporativos até infraestruturas críticas de energia, saúde e comunicações. Sistemas de Detecção de Intrusão (IDS — Intrusion Detection Systems) são a principal linha de defesa ativa contra essas ameaças, e a automação dessa detecção por meio de aprendizado de máquina representa a fronteira atual da área.

As características de um ataque DoS se misturam às do tráfego legítimo em regiões do espaço de atributos que nenhuma linha reta consegue separar adequadamente. Assim sendo, uma rede Perceptron não é capaz de solucionar esse problema. Assim sendo, o Perceptron Multicamadas (MLP — Multilayer Perceptron), com suas camadas ocultas e funções de ativação não lineares, supera essa limitação, tornando-se uma escolha natural para este tipo de problema.


## Etapas do Projeto

1. Exploração e verificação da qualidade dos dados;
2. Validação das premissas do modelo;
3. Seleção de atributos;
4. Normalização/padronização e divisão dos dados;
5. Ajuste do modelo (treinamento);
6. Avaliação;
7. Relatório.

## Etapa 1 – Exploração e Verificação da Qualidade dos Dados

**CORRIGIR** **CORRIGIR** **CORRIGIR**

**1. Obter e inspecionar o dataset**

**2. Codificação one-hot**

**3. Binarização**

**4. Verificar a qualidade dos dados**
- **Valores ausentes:** `df.isnull().sum()` – tipicamente nenhum. Se houver, imputar ou remover.
- **Atributos constantes:** Calcular o desvio padrão; remover qualquer preditor com `std ≈ 0`.
- **Duplicatas:** `df.duplicated().sum()` e remoção, se existirem.

**5. Análise univariada**
- Plotar histogramas para cada um dos 34 atributos.
- Plotar boxplots para detectar outliers severos.

**6. Análise bivariada / multivariada**
- Matriz de correlação de Pearson e mapa de calor.
- Correlação preditor–target: gráfico de barras.
- Boxplots por classe (good vs bad) para as features mais correlacionadas.

**7. Resumo**
- Resumo da qualidade (missings, constantes, outliers).
- Lista de atributos removidos (se houver).
- Matriz de correlação comentada.
- Histogramas e boxplots para o relatório.


In [254]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# Configurações de visualização
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [255]:
# sklearn imports
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    r2_score
)
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold

### 1.1 Obtain and inspect the dataset

As duas últimas colunas (attack_type e difficulty_level) se referem a saída, sendo a última descartada

In [256]:
col_names = [
"duration", "protocol_type", "service", "flag", "src_bytes",
"dst_bytes", "land", "wrong_fragment", "urgent", "hot",
"num_failed_logins", "logged_in", "num_compromised", "root_shell",
"su_attempted", "num_root", "num_file_creations", "num_shells",
"num_access_files", "num_outbound_cmds", "is_host_login",
"is_guest_login", "count", "srv_count", "serror_rate",
"srv_serror_rate", "rerror_rate", "srv_rerror_rate", "same_srv_rate",
"diff_srv_rate", "srv_diff_host_rate", "dst_host_count",
"dst_host_srv_count", "dst_host_same_srv_rate",
"dst_host_diff_srv_rate", "dst_host_same_src_port_rate",
"dst_host_srv_diff_host_rate", "dst_host_serror_rate",
"dst_host_srv_serror_rate", "dst_host_rerror_rate",
"dst_host_srv_rerror_rate", "attack_type", "difficulty_level"
]

# Dados de treino: KDDTrain+.txt
data_train = pd.read_csv("data/archive/KDDTrain+.txt", sep=',', header=None, names=col_names)
data_train = data_train.drop(columns=["difficulty_level"])

# Dados de teste: KDDTest+.txt
data_test = pd.read_csv("data/archive/KDDTest+.txt", sep=',', header=None, names=col_names)
data_test = data_test.drop(columns=["difficulty_level"])

In [257]:
# Visualização do dataset
data_train.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,attack_type
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,25,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal
1,0,udp,other,SF,146,0,0,0,0,0,...,1,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal
2,0,tcp,private,S0,0,0,0,0,0,0,...,26,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune
3,0,tcp,http,SF,232,8153,0,0,0,0,...,255,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal
4,0,tcp,http,SF,199,420,0,0,0,0,...,255,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal


In [258]:
# informações do conjunto de treinamento
data_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 125973 entries, 0 to 125972
Data columns (total 42 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   duration                     125973 non-null  int64  
 1   protocol_type                125973 non-null  str    
 2   service                      125973 non-null  str    
 3   flag                         125973 non-null  str    
 4   src_bytes                    125973 non-null  int64  
 5   dst_bytes                    125973 non-null  int64  
 6   land                         125973 non-null  int64  
 7   wrong_fragment               125973 non-null  int64  
 8   urgent                       125973 non-null  int64  
 9   hot                          125973 non-null  int64  
 10  num_failed_logins            125973 non-null  int64  
 11  logged_in                    125973 non-null  int64  
 12  num_compromised              125973 non-null  int64  
 13  root_shell

In [259]:
# informações estatística do conjunto de treinamento
# Valores categóricos (String atrapalham nessa etapa)
data_train.describe()

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate
count,125973.00000,1.259730e+05,1.259730e+05,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,...,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000
mean,287.14465,4.556674e+04,1.977911e+04,0.000198,0.022687,0.000111,0.204409,0.001222,0.395736,0.279250,...,182.148945,115.653005,0.521242,0.082951,0.148379,0.032542,0.284452,0.278485,0.118832,0.120240
std,2604.51531,5.870331e+06,4.021269e+06,0.014086,0.253530,0.014366,2.149968,0.045239,0.489010,23.942042,...,99.206213,110.702741,0.448949,0.188922,0.308997,0.112564,0.444784,0.445669,0.306557,0.319459
min,0.00000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.00000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,82.000000,10.000000,0.050000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.00000,4.400000e+01,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,255.000000,63.000000,0.510000,0.020000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.00000,2.760000e+02,5.160000e+02,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,...,255.000000,255.000000,1.000000,0.070000,0.060000,0.020000,1.000000,1.000000,0.000000,0.000000
max,42908.00000,1.379964e+09,1.309937e+09,1.000000,3.000000,3.000000,77.000000,5.000000,1.000000,7479.000000,...,255.000000,255.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [260]:
# informações do conjunto de teste
data_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 22544 entries, 0 to 22543
Data columns (total 42 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   duration                     22544 non-null  int64  
 1   protocol_type                22544 non-null  str    
 2   service                      22544 non-null  str    
 3   flag                         22544 non-null  str    
 4   src_bytes                    22544 non-null  int64  
 5   dst_bytes                    22544 non-null  int64  
 6   land                         22544 non-null  int64  
 7   wrong_fragment               22544 non-null  int64  
 8   urgent                       22544 non-null  int64  
 9   hot                          22544 non-null  int64  
 10  num_failed_logins            22544 non-null  int64  
 11  logged_in                    22544 non-null  int64  
 12  num_compromised              22544 non-null  int64  
 13  root_shell                 

In [261]:
# informações estatística do conjunto de teste
# Valores categóricos (String atrapalham nessa etapa)
data_test.describe()

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate
count,22544.000000,2.254400e+04,2.254400e+04,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,...,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000
mean,218.859076,1.039545e+04,2.056019e+03,0.000311,0.008428,0.000710,0.105394,0.021647,0.442202,0.119899,...,193.869411,140.750532,0.608722,0.090540,0.132261,0.019638,0.097814,0.099426,0.233385,0.226683
std,1407.176612,4.727864e+05,2.121930e+04,0.017619,0.142599,0.036473,0.928428,0.150328,0.496659,7.269597,...,94.035663,111.783972,0.435688,0.220717,0.306268,0.085394,0.273139,0.281866,0.387229,0.400875
min,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,121.000000,15.000000,0.070000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,5.400000e+01,4.600000e+01,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,255.000000,168.000000,0.920000,0.010000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,2.870000e+02,6.010000e+02,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,...,255.000000,255.000000,1.000000,0.060000,0.030000,0.010000,0.000000,0.000000,0.360000,0.170000
max,57715.000000,6.282565e+07,1.345927e+06,1.000000,3.000000,3.000000,101.000000,4.000000,1.000000,796.000000,...,255.000000,255.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [262]:
# Copia dos dados para segurança nas próximas etapas
df_train = data_train.copy()
df_test = data_test.copy()

### 1.2 Codificação one-hot
Codificação one-hot paara três dados categóricos (protocol_type, service, flag)

**ONE-HOT ENCODING**

1. Converte texto em números
2. Sem criar ordem artificial entre categorias (tcp != udp)
3. Cada categoria independente 
4. O modelo consegue desligar uma categoria quando ela não apareca (se for tcp, as colunas udp e icmp = 0)

DADOS ORIGINAIS (categorias como texto):
- protocol_type: tcp
- service: ftp_data
- flag: SF

APÓS ONE-HOT ENCODING (valores binários):

**RESUMO DO ENCODING:**
- Valores: APENAS 0 e 1
- Por linha: EXATAMENTE 1 em cada categoria
- Total: 42 colunas originais → 123 colunas finais (+81 codificadas)

**Exemplo:**

Converte 1 coluna categórica em 3 colunas binárias:

Original: protocol_type = "tcp"

Após one-hot:
- protocol_type_tcp:  1 X
- protocol_type_udp:  0
- protocol_type_icmp: 0

Original: protocol_type = "udp"

Após one-hot:
- protocol_type_tcp:  0
- protocol_type_udp:  1 X
- protocol_type_icmp: 0

**O problema:**
Treino e Teste têm 123 colunas iguais, mas têm valores diferentes dentro delas.

- No treino apareceram 70 tipos de serviço diferentes
- No teste apareceram apenas 64 tipos, **6 serviços do treino não existem no teste (harvest, http_2784, aol, etc)**

Isso significa que no teste, essas 6 colunas são sempre zero
Nenhuma conexão usa esses serviços no teste
É como se no teste nunca houve ataque/conexão daquele tipo

VALIDAR BALANCEAMENTO DE NORMAL VS ATAQUE
Para usar: y_binary = y_binary_1


In [263]:
# One-hot encode para: protocol_type, service e flag

# IMPORTANTE: Fazer get_dummies no treino primeiro, depois alinhar o teste
df_train_encoded = pd.get_dummies(df_train, columns=['protocol_type', 'service', 'flag'], dtype=int)
df_test_encoded = pd.get_dummies(df_test, columns=['protocol_type', 'service', 'flag'], dtype=int)

# Alinhar as colunas do teste com o treino (adicionar colunas faltantes com 0)
df_test_encoded = df_test_encoded.reindex(columns=df_train_encoded.columns, fill_value=0)

In [264]:
df_train_encoded.head()

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,flag_REJ,flag_RSTO,flag_RSTOS0,flag_RSTR,flag_S0,flag_S1,flag_S2,flag_S3,flag_SF,flag_SH
0,0,491,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
1,0,146,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3,0,232,8153,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,1,0
4,0,199,420,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,1,0


### 1.3 Binarização de rótulo

Binarização de rótulo: 
- 0 - será Tráfego normal
- 1 - Qualquer tipo de ataque



In [265]:
# BINARIZAÇÃO DO TARGET 
# compara cada valor com normale faz o cast para int: False = 0 e True = 1
target_binary = (df_train_encoded['attack_type'] != 'normal').astype(int) # diferente de normal então é um ataque
target_test_binary = (df_test_encoded['attack_type'] != 'normal').astype(int)


In [266]:
print("Distribuição TREINO:")
print(f"  Normal: {(target_binary == 0).sum()} amostras")
print(f"  Ataque: {(target_binary == 1).sum()} amostras")

print(f"\n Distribuição TESTE:")
print(f"  Normal: {(target_test_binary  == 0).sum()} amostras")
print(f"  Ataque: {(target_test_binary  == 1).sum()} amostras")

Distribuição TREINO:
  Normal: 67343 amostras
  Ataque: 58630 amostras

 Distribuição TESTE:
  Normal: 9711 amostras
  Ataque: 12833 amostras


### 1.4 Verificar a Qualidade dos dados / Data quality checks

### 1.5 Análise univariada / Univariate analysis

### 1.6 Análise bivariada e multivariada / Bivariate and multivariate analysis

#### 1.6.1 Matriz de correlação / Correlation Matrix

#### Correlação entre Preditores e Variável Alvo (PT-BR) / Preditor-Target Correlation 

#### Potencial discriminativo / Discriminative potential 

### 1.7 Resumo

## Etapa 2 – Validação das Premissas do Modelo 

## Etapa 3 – Seleção de Atributos


In [267]:
# Remover coluna target do treino, manter 122 features
X = df_train_encoded.drop(columns=['attack_type'])
y = target_binary

print(X.shape)
print(y.shape)

(125973, 122)
(125973,)


In [268]:
X_over_test = df_test_encoded.drop(columns=['attack_type'])
y_over_test = target_test_binary

## Etapa 4 – Normalização/Padronização e Divisão dos Dados




### 4.1 - Normalização
- Separar 80% para treino e 20% para validação (sem embaralhamento após a divisão)

**Motivo do `stratify=y`:**
- Garante que proporção de classes (0 vs 1) se mantém no treino e teste
- Evita desbalanceamento acidental

In [269]:
# Divisão do conjunto de treino
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    # random_state=42, 
    stratify=y
)

In [270]:
print(f"  X_train: {X_train.shape}")
print(f"  X_test:  {X_test.shape}")
print(f"  y_train: {y_train.shape}")
print(f"  y_test:  {y_test.shape}")

  X_train: (100778, 122)
  X_test:  (25195, 122)
  y_train: (100778,)
  y_test:  (25195,)


### 4.2 Padronização

- Aplicar padronização Z‑score (`StandardScaler`) após a divisão treino‑teste.
- Ajustar o scaler somente no conjunto de treino; transformar treino e teste com os mesmos parâmetros.
- Justificativa algébrica: a regra Delta atualiza pesos com `w = w + η (y - ŷ) x`. Features em escalas diferentes tornam a superfície de erro alongada, exigindo η muito pequeno e causando convergência lenta. A padronização equaliza as variâncias, permitindo um η estável.


In [271]:
scaler = StandardScaler()

# Fit no treino, transform em ambos
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Conversão para DataFrame (opcional, para visualização)
# X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
# X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

In [272]:
# Fit do conjunto de teste
X_over_scaled = scaler.fit_transform(X_over_test)

# Conversão para DataFrame (opcional, para visualização)
# X_over_scaled = pd.DataFrame(X_over_scaled, columns=X.columns)

In [ ]:
print(type(y)) 
print(type(y_train))

<class 'pandas.Series'>
<class 'pandas.Series'>


In [274]:
# Convertendo para tensores do Pytorch
X_train = torch.tensor(X_train_scaled, dtype = torch.float32)
X_test = torch.tensor(X_test_scaled, dtype = torch.float32)
y_train = torch.tensor(y_train.values, dtype = torch.float32)
y_test = torch.tensor(y_test.values, dtype = torch.float32)

In [276]:
print(type(y)) 
print(type(y_train))

<class 'pandas.Series'>
<class 'torch.Tensor'>


## Etapa 5 – Ajuste do Modelo (Adaline)



In [275]:
# hiperparâmetros solicitados
etas = []
epochs = 0

## Etapa 6 – Avaliação


### **Conclusão**

## Etapa 7 – Relatório Final


1. Preparação dos dados: realizado na etapa 1 ao 4
2. Implementação da MPL em PyTorch: realizado na etapa 5
3. Treinamento e monitoramento: realizado na 5
4. Avaliação final: realizado na etapa 6
